In [1]:
import os
import pandas as pd
import numpy as np

# Ensure target directories exist
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# Seed for reproducibility
np.random.seed(42)

# Templates to synthesize realistic movie reviews across the 3 target classes
templates = {
    0: [  # Negative
        "Absolute garbage. The plot was non-existent and the acting was terrible.",
        "A complete waste of time and money. Do not watch this film.",
        "Terrible directing, messy editing, and incredibly boring execution.",
        "I hated every minute of this movie. The characters were obnoxious.",
        "Predictable, lazy writing, and awful special effects."
    ],
    1: [  # Neutral
        "The movie was okay. It had decent acting but a slow, average script.",
        "Nothing special. It serves as a passable popcorn flick but is highly forgettable.",
        "An average experience. Some scenes were great, others dragged on forever.",
        "Mediocre execution. The performances were fine, but the plot was standard.",
        "It was watchable, but don't expect anything ground-breaking or deep."
    ],
    2: [  # Positive
        "An absolute masterpiece! Brilliant directing and stellar performances throughout.",
        "Incredible cinematography and a deeply moving storyline. Highly recommend.",
        "Fantastic movie! Engaging from start to finish with great character development.",
        "A brilliant piece of filmmaking. The editing and musical score were perfect.",
        "Outstanding acting and a gripping plot that keeps you on the edge of your seat."
    ]
}

data = []
for label, reviews in templates.items():
    # Expand templates with random padding variation to simulate 500 distinct samples per class
    for i in range(500):
        base_text = np.random.choice(reviews)
        variant = f"{base_text} Variation context index code-{i}."
        data.append({"review_text": variant, "sentiment_label": label})

df = pd.DataFrame(data)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Export raw dataset
raw_path = "../data/raw/canned_reviews_3class.csv"
df.to_csv(raw_path, index=False)
print(f"Dataset generated with shape {df.shape} and exported to {raw_path}")
print("\nClass Distribution:\n", df["sentiment_label"].value_counts())

Dataset generated with shape (1500, 2) and exported to ../data/raw/canned_reviews_3class.csv

Class Distribution:
 sentiment_label
2    500
0    500
1    500
Name: count, dtype: int64


In [2]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append("../")
from src.data_preprocessing import TextPreprocessor

# Load generated data
df = pd.read_csv("../data/raw/canned_reviews_3class.csv")

# Initialize custom preprocessor
processor = TextPreprocessor()
df["cleaned_text"] = df["review_text"].apply(processor.clean_text)

# Perform stratified split to protect 3-class distribution balance
X_train, X_test, y_train, y_test = train_test_split(
    df["cleaned_text"], 
    df["sentiment_label"], 
    test_size=0.20, 
    stratify=df["sentiment_label"], 
    random_state=42
)

# Export splits for cross-engine documentation tracking
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print(f"Train subset dimensions: {X_train.shape}")
print(f"Test subset dimensions: {X_test.shape}")

Train subset dimensions: (1200,)
Test subset dimensions: (300,)


In [4]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# Construct a unified pipeline - multi_class removed for scikit-learn 1.8+ compatibility
baseline_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(solver="lbfgs", random_state=42))
])

print("Training multi-class baseline model...")
baseline_pipeline.fit(X_train, y_train)

# Evaluate model performance
predictions = baseline_pipeline.predict(X_test)
print("\n--- Model Evaluation Diagnostics ---")
print(classification_report(y_test, predictions, target_names=["Negative", "Neutral", "Positive"]))

# Serialize complete pipeline to artifacts directory
model_export_path = "../models/baseline_3class.pkl"
joblib.dump(baseline_pipeline, model_export_path)
print(f"\nBaseline pipeline successfully serialized to: {model_export_path}")

Training multi-class baseline model...

--- Model Evaluation Diagnostics ---
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       100
     Neutral       1.00      1.00      1.00       100
    Positive       1.00      1.00      1.00       100

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Baseline pipeline successfully serialized to: ../models/baseline_3class.pkl


In [1]:
import pandas as pd
from transformers import RobertaTokenizerFast

# 1. Load the active dataset from your drive
DATA_FILE = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
df = pd.read_csv(DATA_FILE)

print("==================================================")
print("          DIAGNOSTIC STEP 1: DATA INTEGRITY       ")
print("==================================================")

# Check structural sanity
print(f"Dataset Shape: {df.shape}")
print(f"Columns Found: {list(df.columns)}")
print(f"Unique Value Counts in 'label' column:")
print(df['label'].value_counts())
print(f"Data types:\n{df.dtypes}")

print("\n==================================================")
print("          DIAGNOSTIC STEP 2: SAMPLE ALIGNMENT     ")
print("==================================================")

# Print out the first 5 rows to visually confirm text matches the numeric intent
for idx, row in df.head(5).iterrows():
    print(f"Row [{idx}] | Label: {row['label']} | Text: \"{str(row['text'])[:90]}...\"")

print("\n==================================================")
print("          DIAGNOSTIC STEP 3: TOKENIZER MAPPING    ")
print("==================================================")

# Initialize tokenizer and verify raw tensor transformation
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
sample_text = str(df['text'].iloc[0])
sample_label = df['label'].iloc[0]

encodings = tokenizer(sample_text, truncation=True, padding='max_length', max_length=15)

print(f"Raw String : \"{sample_text[:100]}...\"")
print(f"Target Label: {sample_label}")
print(f"Tokens      : {tokenizer.convert_ids_to_tokens(encodings['input_ids'])}")
print(f"Input IDs   : {encodings['input_ids']}")
print(f"Attention   : {encodings['attention_mask']}")

          DIAGNOSTIC STEP 1: DATA INTEGRITY       
Dataset Shape: (1200, 2)
Columns Found: ['text', 'label']
Unique Value Counts in 'label' column:
label
2    400
0    400
1    400
Name: count, dtype: int64
Data types:
text       str
label    int64
dtype: object

          DIAGNOSTIC STEP 2: SAMPLE ALIGNMENT     
Row [0] | Label: 2 | Text: " Awwwwwww. You two are the cutest.  And gods, I LOVE your hair...."
Row [1] | Label: 2 | Text: " Hey girl, yeah I did..thanks a bunch!! I haven`t started downloading them yet...I totally..."
Row [2] | Label: 0 | Text: "it keeps bugging out whenever i try to do anything. it wont stop doing ghe tutorial and wo..."
Row [3] | Label: 1 | Text: "Portland city politics may undo baseball park http://tinyurl.com/lpjquj..."
Row [4] | Label: 0 | Text: "My last day with my favorite teacher.....im quite sad..."

          DIAGNOSTIC STEP 3: TOKENIZER MAPPING    


Raw String : " Awwwwwww. You two are the cutest.  And gods, I LOVE your hair...."
Target Label: 2
Tokens      : ['<s>', 'A', 'w', 'w', 'w', 'w', 'w', 'w', 'w', '.', 'Y', 'o', 'u', 't', '</s>']
Input IDs   : [0, 250, 605, 605, 605, 605, 605, 605, 605, 4, 975, 139, 257, 90, 2]
Attention   : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [2]:
import gc
import torch
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from torch.optim import AdamW

# 1. Force clear VRAM handles
if 'model' in globals(): del model
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")

# 2. Load and tokenize the balanced baseline dataset
df = pd.read_csv(r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv")

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base", local_files_only=True)
encodings = tokenizer(df['text'].astype(str).tolist(), truncation=True, padding=True, max_length=128, return_tensors="pt")
labels = torch.tensor(df['label'].tolist())

# 3. Build a clean, transparent PyTorch DataLoader pipeline
dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'], labels)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 4. Initialize model weights explicitly
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=3)
model.to(device)

# Standard stable optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

print("\n--- RUNNING 5-STEP GRADIENT MONITORING LOOP ---")
model.train()

for step, batch in enumerate(loader):
    if step >= 5: 
        break
        
    # Unpack tensors and push directly to your RTX 4050
    input_ids, attention_mask, batch_labels = [b.to(device) for b in batch]
    
    optimizer.zero_grad()
    
    # Forward pass through the network
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=batch_labels)
    loss = outputs.loss
    
    # Backward pass to calculate raw mathematical gradients
    loss.backward()
    
    # Isolate the exact gradient norm of the final classification layer output
    grad_norm = 0.0
    if hasattr(model.classifier, 'out_proj') and model.classifier.out_proj.weight.grad is not None:
        grad_norm = model.classifier.out_proj.weight.grad.norm().item()
        
    print(f"Step [{step}] | Calculated Loss: {loss.item():.4f} | Final Layer Grad Norm: {grad_norm:.6f}")
    
    optimizer.step()

Using device: CUDA


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- RUNNING 5-STEP GRADIENT MONITORING LOOP ---
Step [0] | Calculated Loss: 1.0731 | Final Layer Grad Norm: 1.620435
Step [1] | Calculated Loss: 1.1084 | Final Layer Grad Norm: 0.868597
Step [2] | Calculated Loss: 1.0875 | Final Layer Grad Norm: 0.517123
Step [3] | Calculated Loss: 1.1569 | Final Layer Grad Norm: 2.220991
Step [4] | Calculated Loss: 1.0789 | Final Layer Grad Norm: 1.024486


In [3]:
import os
import gc
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from torch.optim import AdamW

# 1. Reset VRAM to prevent memory fragmentation
if 'model' in globals(): del model
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA reset complete. Running training on: {device.upper()}")

# 2. Load and split data into 80/20 splits
df = pd.read_csv(r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv")

X_train_text, X_val_text, y_train_labels, y_val_labels = train_test_split(
    df['text'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    stratify=df['label'].tolist(),
    random_state=42
)

# 3. Tokenize both sets
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base", local_files_only=True)

train_enc = tokenizer(X_train_text, truncation=True, padding=True, max_length=128, return_tensors="pt")
val_enc = tokenizer(X_val_text, truncation=True, padding=True, max_length=128, return_tensors="pt")

train_loader = DataLoader(TensorDataset(train_enc['input_ids'], train_enc['attention_mask'], torch.tensor(y_train_labels)), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(val_enc['input_ids'], val_enc['attention_mask'], torch.tensor(y_val_labels)), batch_size=16, shuffle=False)

# 4. Initialize model weights
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=3)
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

print(f"\n--- STARTING RAW FINE-TUNING LOOP (3 EPOCHS) ---")

for epoch in range(3):
    model.train()
    total_train_loss = 0
    
    for batch in train_loader:
        input_ids, attention_mask, batch_labels = [b.to(device) for b in batch]
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=batch_labels)
        loss = outputs.loss
        
        loss.backward()
        # Explicit gradient clipping to prevent anomalies
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation phase
    model.eval()
    all_preds = []
    all_labels = []
    total_val_loss = 0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, batch_labels = [b.to(device) for b in batch]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=batch_labels)
            
            total_val_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(batch_labels.cpu().numpy())
            
    avg_val_loss = total_val_loss / len(val_loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"Epoch {epoch+1}/3 | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {epoch_acc:.4f} | Val F1: {epoch_f1:.4f}")

# 5. Save the successfully trained model assets
OUTPUT_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_base_academic_3class"
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel successfully calibrated and saved to: {OUTPUT_DIR}")

CUDA reset complete. Running training on: CUDA


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- STARTING RAW FINE-TUNING LOOP (3 EPOCHS) ---
Epoch 1/3 | Train Loss: 1.1027 | Val Loss: 1.1003 | Val Acc: 0.3333 | Val F1: 0.1667
Epoch 2/3 | Train Loss: 1.1034 | Val Loss: 1.0987 | Val Acc: 0.3375 | Val F1: 0.2813
Epoch 3/3 | Train Loss: 1.1057 | Val Loss: 1.0988 | Val Acc: 0.3333 | Val F1: 0.1667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model successfully calibrated and saved to: F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_base_academic_3class


In [4]:
import pandas as pd
import torch
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
import torch.nn.functional as F

print("--- LOADING PRODUCTION-READY ROBERTA PIPELINE ---")

# 1. Load the pre-trained 3-class RoBERTa model from CardiffNLP
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# 2. Grab a few real rows from your cached academic benchmark
DATA_FILE = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
df = pd.read_csv(DATA_FILE)
sample_rows = df.head(5)

print("\n==================================================")
print("          PRODUCTION MODEL INFERENCE TEST         ")
print("==================================================")

class_labels = {0: "Negative", 1: "Neutral", 2: "Positive"}

with torch.no_grad():
    for idx, row in sample_rows.iterrows():
        raw_text = str(row['text'])
        actual_label = int(row['label'])
        
        # Tokenize and push to GPU
        inputs = tokenizer(raw_text, return_tensors="pt", truncation=True, max_length=128).to(device)
        
        # Run forward pass
        outputs = model(**inputs)
        
        # Convert raw output logits into clean percentages
        probabilities = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        predicted_label = probabilities.argmax()
        confidence = probabilities[predicted_label] * 100
        
        print(f"👉 Row [{idx}]")
        print(f"Text       : \"{raw_text[:90]}...\"")
        print(f"Actual     : {class_labels[actual_label]}")
        print(f"Prediction : {class_labels[predicted_label]} ({confidence:.1f}% Confidence)")
        print("-" * 50)

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


--- LOADING PRODUCTION-READY ROBERTA PIPELINE ---


RuntimeError: Cannot send a request, as the client has been closed.

In [1]:
import os
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

print("--- LOADING EFfICIENT PRODUCTION PIPELINE ---")

# 1. Target the pre-trained 3-class model
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

try:
    # Explicitly pull fresh instances under a clean session
    tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
    model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    print(f"Success! Model fetched and deployed onto: {device.upper()}")

    # 2. Grab rows from your local cached academic data to verify predictions
    DATA_FILE = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
    df = pd.read_csv(DATA_FILE)
    sample_rows = df.head(5)

    print("\n==================================================")
    print("          PRODUCTION MODEL INFERENCE TEST         ")
    print("==================================================")

    class_labels = {0: "Negative", 1: "Neutral", 2: "Positive"}

    with torch.no_grad():
        for idx, row in sample_rows.iterrows():
            raw_text = str(row['text'])
            actual_label = int(row['label'])
            
            inputs = tokenizer(raw_text, return_tensors="pt", truncation=True, max_length=128).to(device)
            outputs = model(**inputs)
            
            probabilities = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
            predicted_label = probabilities.argmax()
            confidence = probabilities[predicted_label] * 100
            
            print(f"👉 Row [{idx}]")
            print(f"Text       : \"{raw_text[:90]}...\"")
            print(f"Actual     : {class_labels[actual_label]}")
            print(f"Prediction : {class_labels[predicted_label]} ({confidence:.1f}% Confidence)")
            print("-" * 50)

except Exception as e:
    print(f"Initialization stalled: {str(e)}")
    print("If it fails, double-check that your internet connection is active for this first download step.")

--- LOADING EFfICIENT PRODUCTION PIPELINE ---


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Initialization stalled: not enough values to unpack (expected 2, got 1)
If it fails, double-check that your internet connection is active for this first download step.


In [2]:
import os
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("--- OVERWRITING CACHE AND LOADING PRODUCTION PIPELINE ---")

# Target the pre-trained 3-class production model
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

try:
    # force_download=True clears out any corrupted partial files from previous crashes
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, force_download=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, force_download=True)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    print(f"Success! Fresh model downloaded and deployed onto: {device.upper()}")

    # 2. Grab a few real rows from your local cached academic data to verify predictions
    DATA_FILE = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
    df = pd.read_csv(DATA_FILE)
    sample_rows = df.head(5)

    print("\n==================================================")
    print("          PRODUCTION MODEL INFERENCE TEST         ")
    print("==================================================")

    class_labels = {0: "Negative", 1: "Neutral", 2: "Positive"}

    with torch.no_grad():
        for idx, row in sample_rows.iterrows():
            raw_text = str(row['text'])
            actual_label = int(row['label'])
            
            inputs = tokenizer(raw_text, return_tensors="pt", truncation=True, max_length=128).to(device)
            outputs = model(**inputs)
            
            probabilities = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
            predicted_label = probabilities.argmax()
            confidence = probabilities[predicted_label] * 100
            
            print(f"👉 Row [{idx}]")
            print(f"Text       : \"{raw_text[:90]}...\"")
            print(f"Actual     : {class_labels[actual_label]}")
            print(f"Prediction : {class_labels[predicted_label]} ({confidence:.1f}% Confidence)")
            print("-" * 50)

except Exception as e:
    print(f"\nPipeline block stalled: {str(e)}")
    print("If it still fails, make sure your internet connection is active to handle this redownload step.")

--- OVERWRITING CACHE AND LOADING PRODUCTION PIPELINE ---


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpx\_transports\default.py", line 101, in map_httpcore_exceptions
    yield
  File "c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpx\_transports\default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpcore\_sync\connection_pool.py", line 256, in handle_request
    raise exc from None
  File "c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpcore\_sync\connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpcore\_sync\connection.py", line 101, in handle_request


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

    raise exc
  File "c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpcore\_sync\connection.py", line 78, in handle_request
    stream = self._connect(request)
             ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpcore\_sync\connection.py", line 156, in _connect
    stream = stream.start_tls(**kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpcore\_backends\sync.py", line 154, in start_tls
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
    

Success! Fresh model downloaded and deployed onto: CUDA

          PRODUCTION MODEL INFERENCE TEST         
👉 Row [0]
Text       : " Awwwwwww. You two are the cutest.  And gods, I LOVE your hair...."
Actual     : Positive
Prediction : Positive (98.4% Confidence)
--------------------------------------------------
👉 Row [1]
Text       : " Hey girl, yeah I did..thanks a bunch!! I haven`t started downloading them yet...I totally..."
Actual     : Positive
Prediction : Positive (96.8% Confidence)
--------------------------------------------------
👉 Row [2]
Text       : "it keeps bugging out whenever i try to do anything. it wont stop doing ghe tutorial and wo..."
Actual     : Negative
Prediction : Negative (89.5% Confidence)
--------------------------------------------------
👉 Row [3]
Text       : "Portland city politics may undo baseball park http://tinyurl.com/lpjquj..."
Actual     : Neutral
Prediction : Neutral (62.9% Confidence)
--------------------------------------------------
👉 Row [4

In [3]:
import os
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("--- STARTING LIVE TMDB CORPUS SCORING PIPELINE ---")

# 1. Environment & Path Setup
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
LIVE_DATA_PATH = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\tmdb_live_captured.csv"
OUTPUT_DATA_PATH = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\tmdb_scored_output.csv"

if not os.path.exists(LIVE_DATA_PATH):
    raise FileNotFoundError(f"Missing active live data file at: {LIVE_DATA_PATH}")

df_live = pd.read_csv(LIVE_DATA_PATH)
print(f"Loaded live dataset containing {len(df_live)} un-scored user reviews.")

# 2. Re-verify pipeline memory allocations
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

predicted_labels = []
confidence_scores = []
class_labels = {0: "Negative", 1: "Neutral", 2: "Positive"}

print("\nProcessing review sequences through the transformer...")

# 3. Batchless Inference Loop for precise sequence handling
with torch.no_grad():
    for idx, row in df_live.iterrows():
        review_text = str(row['text'])
        
        # Tokenize sequence explicitly for GPU registers
        inputs = tokenizer(review_text, return_tensors="pt", truncation=True, max_length=256).to(device)
        outputs = model(**inputs)
        
        # Calculate scaled probabilities
        probabilities = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred_idx = probabilities.argmax()
        
        predicted_labels.append(class_labels[pred_idx])
        confidence_scores.append(round(float(probabilities[pred_idx]), 4))

# 4. Map results back to the dataframe structure
df_live['model_sentiment'] = predicted_labels
df_live['model_confidence'] = confidence_scores

# 5. Export clean output matrix to drive
df_live.to_csv(OUTPUT_DATA_PATH, index=False)

print("\n==================================================")
print("          SCORING RUN COMPLETION SUMMARY          ")
print("==================================================")
print(f"👉 Total Rows Processed    : {len(df_live)}")
print(f"👉 Output Dataset Saved To : {OUTPUT_DATA_PATH}")
print("\nSentiment Distribution Breakdown:")
print(df_live['model_sentiment'].value_counts())
print("==================================================")

--- STARTING LIVE TMDB CORPUS SCORING PIPELINE ---
Loaded live dataset containing 49 un-scored user reviews.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Processing review sequences through the transformer...

          SCORING RUN COMPLETION SUMMARY          
👉 Total Rows Processed    : 49
👉 Output Dataset Saved To : F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\tmdb_scored_output.csv

Sentiment Distribution Breakdown:
model_sentiment
Positive    34
Negative     8
Neutral      7
Name: count, dtype: int64


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]